In [1]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

In [2]:

from box import Box
from datasetz.core.load_dataset import load_embedded_dataset
from loguru import logger as log
from mlflow import MlflowClient
import mlflow
from mlutils.mlflow.utils import get_run_params, terminate_run, finish_run_and_print_exception
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier
import problexity as px
import sys
import numpy as np
import pandas as pd

from optimalcentroids.optimal_centroids_explainer import run_tree as run_explainer_with_tree_selection, run as run_explainer
from optimalcentroids.optimal_centroids_complexity_measure import run as run_complexity_measure
from optimalcentroids.optimal_centroids import run as run_optimal_centroids
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from mlutils.scikit.utils import is_fitted
from sklearn.metrics import classification_report, confusion_matrix


/Users/bgulowaty/studia/projekty/optimal-centroids/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-04 18:54:08,142	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
DT_PARAMS = {'ccp_alpha': 0.011538226894236229, 'criterion': 'gini', 'max_features': None}
DT_PARAMS_WO_DEPTH = {'criterion': 'gini', 'max_features': None}

In [4]:
log.remove()
log.add(sys.stderr, level="INFO")

1

In [5]:
def calculate_metrics_dt(model):
    return {
        'n_leaves': model.get_n_leaves(),
        'depth': model.get_depth()
    }

In [6]:
def calculate_confusion_matrix(y_pred, y_true):
    tn, fp, fn, tp = confusion_matrix(y_pred, y_true).ravel()

    return {
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'tp': tp
    }

In [7]:
def calculate_simple_ensemble_metrics(model):
    df = pd.DataFrame([calculate_metrics_dt(dt) for dt in model.clf_by_label.values() if is_fitted(dt)])

    median = df.median()
    mean = df.mean()
    max_df = df.max()
    min_df = df.min()

    return {
        'n_leaves_median': median.n_leaves,
        'n_leaves_mean': mean.n_leaves,
        'depth_median': median.depth,
        'depth_mean': mean.depth,
        'depth_max': max_df.depth,
        'n_leaves_max': max_df.n_leaves,
        'depth_min': min_df.depth,
        'n_leaves_min': min_df.n_leaves,

    }

In [8]:
def calculate_simple_ensemble_metrics_simple(model):
    models = pd.DataFrame([m for m in model.clf_by_label.values() if is_fitted(m)])

    return {
        'fitted_models': len(models)
    }

In [9]:
def get_params_and_client(run_id):
    client = MlflowClient(tracking_uri="http://192.168.1.181:5010")
    mlflow.set_tracking_uri("http://192.168.1.181:5010")
    params = get_run_params(run_id, client)
    params.should_take_test = params.should_take_test.lower() == "true"
    params.data_shuffle_random_state = int(params.data_shuffle_random_state)
    return params, client


def get_dataset(params):
    dataset = load_embedded_dataset('keel-binary-fast', params.dataset).encode_x_to_labels()

    x_whole = dataset.x()
    y_whole = dataset.y()

    sss = StratifiedShuffleSplit(random_state=params.data_shuffle_random_state, n_splits=1, test_size=0.5)
    train_idx, test_idx = next(sss.split(x_whole, y_whole))

    if params.should_take_test:
        x_train = x_whole[test_idx]
        y_train = y_whole[test_idx]

        x_test = x_whole[train_idx]
        y_test = y_whole[train_idx]
    else:
        x_train = x_whole[train_idx]
        y_train = y_whole[train_idx]

        x_test = x_whole[test_idx]
        y_test = y_whole[test_idx]

    return x_train, y_train, x_test, y_test


In [10]:
COMPLEXITY_METRICS = {
    'f2': px.f2,
    't4': px.t4,
    'c1': px.c1,
    'n3': px.n3,
    'l2': px.l2,
    'density': px.density,
}

MODELS = {
    "knn": KNeighborsClassifier(n_neighbors=3),
    "bayes": GaussianNB(),
    "dt": DecisionTreeClassifier(random_state=42, **DT_PARAMS),
    "svm": LinearSVC(random_state=42)
}

In [11]:
def optimal_centroid_using_complexity_measure(run_id):
    params, client = get_params_and_client(run_id)
    params = Box(params,  box_recast={
        'subspaces': int,
        'n_gen': int,
        'pop_size': int,
    })
    log.info(params)

    choosen_complexity_metric = COMPLEXITY_METRICS.get(params.complexity_measure)

    model = MODELS.get(params.base_clf)

    # model
    log.info(params)

    try:
        x_train, y_train, x_test, y_test = get_dataset(params)

        models = run_complexity_measure(
            x_train=x_train,
            y_train=y_train,
            complexity_metric=choosen_complexity_metric,
            model=model,
            n_pop=params.pop_size,
            n_gen=params.n_gen,
            n_clf=params.subspaces
        )

        model_max_acc = max(models, key=lambda model: accuracy_score(model.predict(x_train), y_train))

        metrics = calculate_simple_ensemble_metrics_simple(model_max_acc)

        y_pred = model_max_acc.predict(x_test)
        conf_matrix = calculate_confusion_matrix(y_pred, y_test)
        cls_report = classification_report(y_pred, y_test, output_dict=True)

        client.log_dict(run_id, cls_report, 'cls_report')

        for k, v in {**metrics, **conf_matrix}.items():
            client.log_metric(run_id, k, v)

        terminate_run(run_id, client=client)
        log.info("Run finished")
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)

In [12]:
def optimal_centroid(run_id):
    params, client = get_params_and_client(run_id)
    params = Box(params,  box_recast={
        'subspaces': int,
        'n_gen': int,
        'pop_size': int,
    })
    params['max_tree_depth'] = int(params['max_tree_depth']) if 'max_tree_depth' in params else None

    log.info(params)

    try:
        x_train, y_train, x_test, y_test = get_dataset(params)
        

        # model
        models = run_optimal_centroids(x_train, y_train, params.subspaces, params.max_tree_depth, pop_size=params.pop_size, n_gen=params.n_gen, mlflow_client=client, run_id=run_id, tree_params=DT_PARAMS_WO_DEPTH)

        model_max_acc = max(models, key=lambda model: accuracy_score(model.predict(x_train), y_train))

        metrics = calculate_simple_ensemble_metrics(model_max_acc)        

        y_pred = model_max_acc.predict(x_test)
        conf_matrix = calculate_confusion_matrix(y_pred, y_test)
        cls_report = classification_report(y_pred, y_test, output_dict=True)

        client.log_dict(run_id, cls_report, 'cls_report')

        for k, v in {**metrics, **conf_matrix}.items():
            client.log_metric(run_id, k, v)

        terminate_run(run_id, client=client)
        log.info("Run finished")
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)

In [59]:
def explainer_without_tree_selection(run_id):
    params, client = get_params_and_client(run_id)
    params = Box(params,  box_recast={
        'subspaces': int,
        'n_gen': int,
        'pop_size': int,
        'data_shuffle_random_state': int
    })

    log.info(params)

    try:
        x_train, y_train, x_test, y_test = get_dataset(params)

        rf = RandomForestClassifier(n_estimators=32, **DT_PARAMS, random_state=42)
        rf.fit(x_train, y_train)

        # model
        results = run_explainer(rf, params.subspaces, x_train, y_train, pop_size=params.pop_size, n_gen=params.n_gen)

        model_min_complexity =  results.min_complexity.model
        model_max_acc =  results.max_accuracy.model

        max_acc_metrics = calculate_simple_ensemble_metrics(model_max_acc)
        min_complexity_metrics = calculate_simple_ensemble_metrics(model_min_complexity)

        max_acc_y_pred = model_max_acc.predict(x_test)
        min_complexity_y_pred = model_min_complexity.predict(x_test)
        
        max_acc_conf_matrix = calculate_confusion_matrix(max_acc_y_pred, y_test)
        max_acc_cls_report = classification_report(max_acc_y_pred, y_test, output_dict=True)

        min_complexity_conf_matrix = calculate_confusion_matrix(min_complexity_y_pred, y_test)
        min_complexity_cls_report = classification_report(min_complexity_y_pred, y_test, output_dict=True)

        client.log_dict(run_id, max_acc_cls_report, 'max_acc_cls_report')
        client.log_dict(run_id, min_complexity_cls_report, 'min_complexity_cls_report')

        for k, v in {**max_acc_conf_matrix, **max_acc_metrics}.items():
            client.log_metric(run_id, f"max_acc_{k}", v)

        for k, v in {**min_complexity_conf_matrix, **min_complexity_metrics}.items():
            client.log_metric(run_id, f"min_complexity_{k}", v)

        terminate_run(run_id, client=client)
        log.info("Run finished")
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)

In [60]:
def explainer_with_tree_selection(run_id):
    params, client = get_params_and_client(run_id)
    params = Box(params,  box_recast={
        'subspaces': int,
        'n_gen': int,
        'pop_size': int,
        'data_shuffle_random_state': int
    })

    log.info(params)

    try:
        x_train, y_train, x_test, y_test = get_dataset(params)

        rf = RandomForestClassifier(n_estimators=32, **DT_PARAMS, random_state=42)
        rf.fit(x_train, y_train)

        # model
        print(params)
        results = run_explainer_with_tree_selection(rf, params.subspaces, x_train, y_train, pop_size=params.pop_size, n_gen=params.n_gen)

        model_min_complexity =  results.min_complexity.model
        model_max_acc =  results.max_accuracy.model

        max_acc_metrics = calculate_simple_ensemble_metrics(model_max_acc)
        min_complexity_metrics = calculate_simple_ensemble_metrics(model_min_complexity)

        max_acc_y_pred = model_max_acc.predict(x_test)
        min_complexity_y_pred = model_min_complexity.predict(x_test)

        max_acc_conf_matrix = calculate_confusion_matrix(max_acc_y_pred, y_test)
        max_acc_cls_report = classification_report(max_acc_y_pred, y_test, output_dict=True)

        min_complexity_conf_matrix = calculate_confusion_matrix(min_complexity_y_pred, y_test)
        min_complexity_cls_report = classification_report(min_complexity_y_pred, y_test, output_dict=True)

        client.log_dict(run_id, max_acc_cls_report, 'max_acc_cls_report')
        client.log_dict(run_id, min_complexity_cls_report, 'min_complexity_cls_report')

        for k, v in {**max_acc_conf_matrix, **max_acc_metrics}.items():
            client.log_metric(run_id, f"max_acc_{k}", v)

        for k, v in {**min_complexity_conf_matrix, **min_complexity_metrics}.items():
            client.log_metric(run_id, f"min_complexity_{k}", v)

        terminate_run(run_id, client=client)
        log.info("Run finished")
    except Exception as e:
        finish_run_and_print_exception(run_id, e, client = client)